# 16 Null Handling

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Find, count, fill, replace and drop nulls with <code>isNull</code>, <code>na.fill</code>, <code>na.drop</code>, <code>na.replace</code> and <code>coalesce</code>, and understand how nulls behave in comparisons, aggregates, joins and sorting.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
Null means "unknown", and Spark follows SQL's three-valued logic: true, false or unknown. Nulls are behind many silent bugs: filters that drop rows, joins that lose matches, averages that look too high, and <code>concat</code> that blanks out a column. Handling nulls deliberately is a sign of a careful data engineer, and it is tested in both interviews and the certification exam.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Null rules to remember</b>
<ul><li>Any comparison with null (<code>=</code>, <code>!=</code>, <code>&gt;</code>, ...) returns <b>null</b>, not true or false</li><li><code>filter</code> and <code>when</code> treat null as <b>not true</b></li><li><code>null AND false</code> is false, <code>null OR true</code> is true, anything else with null is null</li><li>Aggregates (<code>sum</code>, <code>avg</code>, <code>min</code>, <code>max</code>, <code>count(col)</code>) <b>ignore</b> nulls, while <code>count(*)</code> counts every row</li><li><code>groupBy</code> and <code>distinct</code> treat all nulls as <b>one</b> group</li><li>Joins on <code>=</code> <b>never match</b> null keys. Use <code>&lt;=&gt;</code> / <code>eqNullSafe</code> to match them</li><li>Arithmetic and <code>concat</code> with null return null. <code>coalesce(a, b, ...)</code> returns the first non-null value</li><li>Ascending sort puts nulls first, descending puts them last</li></ul>
</div>

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A loan company calculates the average income of applicants for a risk model. 20% of applicants left income blank. <code>avg(income)</code> silently ignores those rows, so the average describes only the applicants who answered, and the model looks better than reality. The team must decide explicitly: exclude, impute (fill with a median per segment), or flag missing income as its own feature.
</div>

## Syntax

```python
from pyspark.sql import functions as F

df.filter(F.col("x").isNull())                 # find nulls
df.na.fill(0, subset=["qty"])                   # fill (alias: fillna)
df.na.fill({"qty": 0, "city": "unknown"})
df.na.drop(how="any", subset=["id", "email"])   # drop rows (alias: dropna)
df.na.replace(["N/A", ""], None, subset=["city"])  # replace values (alias: replace)
F.coalesce("mobile", "home_phone", F.lit("none"))
F.col("a").eqNullSafe(F.col("b"))               # null-safe equality, SQL: a <=> b
```

## Setup: sample data

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates a customer table with nulls and "fake nulls" (<code>""</code>, <code>"N/A"</code>, <code>"null"</code> as text).
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A 7-row table. Several columns have nulls, and <code>city</code> also contains <code>N/A</code> and empty strings.
</div>

In [0]:
from pyspark.sql import functions as F

customers = spark.createDataFrame(
    [
        (1, "Amal", "amal@mail.com", "9000000001", None, "Chennai", 52000.0, 3),
        (2, "John", None, None, "02012345678", "London", None, 0),
        (3, None, "x@mail.com", "9000000003", None, "N/A", 61000.0, None),
        (4, "Vivek", "vivek@mail.ae", None, None, "", 75000.0, 7),
        (5, "Priya", None, None, None, None, None, None),
        (6, "Sara", "sara@mail.ae", "0500000006", "042000000", "Dubai", 48000.0, 2),
        (None, None, None, None, None, "null", None, None),
    ],
    "customer_id INT, name STRING, email STRING, mobile STRING, home_phone STRING, city STRING, income DOUBLE, orders INT",
)
display(customers)

## 1. Finding and counting nulls

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Counts nulls in every column in one pass, and lists rows with a null <code>email</code>.<br><br>
<b>Why it matters</b><br>A null profile is the first data quality check on any new dataset. Doing it in <b>one</b> aggregation is much cheaper than one <code>filter().count()</code> per column.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A one-row table of null counts per column, for example <code>email</code> = 3 and <code>income</code> = 3. Then the 3 customers without an email.
</div>

In [0]:
null_counts = customers.select([
    F.count(F.when(F.col(c).isNull(), 1)).alias(c) for c in customers.columns
])
null_counts.show()

customers.filter(F.col("email").isNull()).select("customer_id", "name", "email").show()

## 2. Fake nulls: empty strings and placeholder text

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Turns <code>""</code>, <code>"N/A"</code> and <code>"null"</code> into real nulls with <code>na.replace</code>, then recounts.<br><br>
<b>Why it matters</b><br>CSV files and source systems often write missing values as text. They are <b>not</b> null to Spark, so <code>isNull</code> misses them and <code>count(col)</code> counts them. Standardize them at the start of the Silver layer.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Before: <code>city</code> has 1 null. After replacing placeholders: 4 nulls (rows 3, 4, 5 and 7).
</div>

In [0]:
print("city nulls before:", customers.filter(F.col("city").isNull()).count())

standardized = customers.na.replace(["", "N/A", "null"], None, subset=["city"])
print("city nulls after :", standardized.filter(F.col("city").isNull()).count())

# The same with a reusable expression that also trims spaces and ignores case.
fake_nulls = ["", "n/a", "null", "none", "-"]
clean_city = F.when(F.lower(F.trim("city")).isin(fake_nulls), None).otherwise(F.trim("city"))
customers.select("city", clean_city.alias("city_clean")).show()

## 3. Filling nulls: `na.fill`, `coalesce`, and computed defaults

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Fills nulls with constants per column, picks the first available phone number with <code>coalesce</code>, and imputes income with the median.<br><br>
<b>Why it matters</b><br>Different columns need different strategies: 0 for counts, <code>"unknown"</code> for categories, a statistic for measures, or the next available field.
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Watch out</b><br>
<code>na.fill(value)</code> only fills columns whose <b>type matches</b> the value. <code>fill(0)</code> fills numeric columns, <code>fill("x")</code> fills string columns, and other columns are silently left alone.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>orders</code> nulls become 0 and <code>name</code>/<code>city</code> nulls become <code>unknown</code>. <code>contact_phone</code> takes the mobile, else the home phone, else <code>no phone</code>. <code>income_filled</code> replaces nulls with the median income (56500.0).
</div>

In [0]:
filled = standardized.na.fill({"orders": 0, "name": "unknown", "city": "unknown"})
filled.select("customer_id", "name", "city", "orders").show()

with_contact = filled.withColumn("contact_phone", F.coalesce("mobile", "home_phone", F.lit("no phone")))
with_contact.select("customer_id", "mobile", "home_phone", "contact_phone").show()

median_income = standardized.agg(F.median("income")).first()[0]
print("Median income:", median_income)
standardized.select("customer_id", "income", F.coalesce("income", F.lit(median_income)).alias("income_filled")).show()

## 4. Dropping rows: `na.drop`

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Drops rows with <code>how="any"</code>, <code>how="all"</code>, <code>thresh</code> and <code>subset</code>.<br><br>
<b>Why it matters</b><br>Rows without a primary key or with no useful data should usually be removed (or quarantined). <code>thresh</code> keeps rows with at least N non-null values.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>any</code> over all columns keeps only row 6 (the only complete row). <code>subset=["customer_id"]</code> removes the row without an ID (6 left). <code>how="all"</code> removes nothing (7 left), because no row is null in <b>every</b> column. Even the last row has the text <code>"null"</code> in <code>city</code>, which is a string, not a null. <code>thresh=5</code> keeps the 5 rows that have at least 5 non-null values.
</div>

In [0]:
print("any (all columns)      :", customers.na.drop(how="any").count())
print("subset customer_id     :", customers.na.drop(subset=["customer_id"]).count())
print("all (every col is null):", customers.na.drop(how="all").count())
print("thresh=5               :", customers.na.drop(thresh=5).count())

## 5. Nulls in comparisons and logic

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Shows the results of comparisons and boolean logic involving null.<br><br>
<b>Why it matters</b><br>This explains why <code>filter(col != 'x')</code> loses null rows, and why <code>NOT IN</code> with a null in the list returns nothing in SQL.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>null = null</code> is null (not true), <code>null &lt;=&gt; null</code> is true, <code>null AND false</code> is false, <code>null OR true</code> is true, <code>1 NOT IN (2, NULL)</code> is null, and <code>null + 1</code> is null.
</div>

In [0]:
spark.sql("""
    SELECT NULL = NULL          AS eq,
           NULL <=> NULL        AS null_safe_eq,
           NULL AND FALSE       AS and_false,
           NULL AND TRUE        AS and_true,
           NULL OR TRUE         AS or_true,
           1 NOT IN (2, NULL)   AS not_in_with_null,
           NULL + 1             AS plus
""").show()

## 6. Nulls in aggregates

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Compares <code>count(*)</code>, <code>count(col)</code>, <code>sum</code>, <code>avg</code>, and an average that treats null as 0.<br><br>
<b>Why it matters</b><br><code>avg</code> ignores nulls, so it is the average of the <b>known</b> values. Whether that is right depends on the business question. Make the choice explicit.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
7 rows, 4 known incomes, <code>avg_known</code> = 59000.0. <code>avg_null_as_zero</code> = 33714.29 (236000 / 7). <code>max</code> and <code>min</code> ignore nulls too.
</div>

In [0]:
customers.agg(
    F.count("*").alias("rows"),
    F.count("income").alias("known_income"),
    F.sum("income").alias("sum_income"),
    F.round(F.avg("income"), 2).alias("avg_known"),
    F.round(F.avg(F.coalesce("income", F.lit(0.0))), 2).alias("avg_null_as_zero"),
    F.max("income").alias("max_income"),
).show()

## 7. Nulls in joins

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Joins two DataFrames on a key that contains nulls, with normal equality and with null-safe equality.<br><br>
<b>Why it matters</b><br>With <code>=</code>, null keys never match, so rows silently fall out of inner joins (or get null partners in outer joins). If null really means "the same unknown value" on both sides, use <code>eqNullSafe</code>. Usually, though, null join keys indicate a data quality problem to fix upstream.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The normal join returns 1 row (<code>IN</code>). The null-safe join returns 2 rows: <code>IN</code> and the null region matched to <code>Unassigned</code>.
</div>

In [0]:
left = spark.createDataFrame([("IN", 100), (None, 50)], "region STRING, sales INT")
right = spark.createDataFrame([("IN", "India"), (None, "Unassigned")], "region STRING, region_name STRING")

print("Normal equality:")
left.join(right, left.region == right.region, "inner").show()

print("Null-safe equality:")
left.join(right, left.region.eqNullSafe(right.region), "inner").show()

## 8. Useful null functions in SQL

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Shows <code>coalesce</code>, <code>nvl</code>, <code>nvl2</code>, <code>nullif</code>, <code>ifnull</code>, <code>zeroifnull</code> and <code>try_divide</code>.<br><br>
<b>Why it matters</b><br>They show up in exam questions and in SQL-heavy codebases. <code>nullif</code> and <code>try_divide</code> are the standard ways to avoid divide-by-zero errors under ANSI mode.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>nvl2</code> returns <code>has orders</code> or <code>no orders</code>. <code>revenue_per_order</code> is null (not an error) when orders is 0 or null.
</div>

In [0]:
customers.createOrReplaceTempView("customers_v")
spark.sql("""
    SELECT customer_id,
           coalesce(mobile, home_phone, 'none')        AS phone,
           nvl(name, 'unknown')                        AS name,
           nvl2(orders, 'has orders', 'no orders')     AS order_flag,
           zeroifnull(orders)                          AS orders_0,
           nullif(city, '')                            AS city,
           try_divide(income, orders)                  AS revenue_per_order,
           income / nullif(orders, 0)                  AS revenue_per_order_v2
    FROM customers_v
""").show()

## Under the hood

```
x = NULL, y = 5

  x = y      -> NULL        x <=> y   -> false
  x > y      -> NULL        x + y     -> NULL
  filter(NULL) drops the row
  sum([5, NULL, 3]) = 8     count(col) = 2     count(*) = 3

Join on a.k = b.k:
  rows with k = NULL never satisfy the condition (NULL = NULL is NULL)
```

<span style="background:#1E9E5A;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Narrow</span> `coalesce`, `na.fill`, `na.drop` and `isNull` are row-level, with no shuffle.

<span style="background:#2F6FED;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Transformation</span> all lazy. `na.fill` and `na.drop` are just `Project` and `Filter` nodes.

**Optimizer bonus:** for inner joins and many filters, Spark automatically adds `isnotnull(key)` filters, because null keys can never match. That's why you see `IsNotNull` in plans you didn't write. Check the plan below for `isnotnull(region)`.

In [0]:
a = spark.range(100).withColumn("region", F.when(F.col("id") % 10 == 0, None).otherwise((F.col("id") % 3).cast("string")))
b = spark.range(3).withColumn("region", F.col("id").cast("string"))
a.join(b, "region").explain()

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: rows vanish after <code>filter(col != value)</code></b><br>
Comparisons with null are null, and the row is dropped. Add <code>| F.col(c).isNull()</code> if those rows should stay.<br><br>
**⛔ Problem: <code>fillna(0)</code> didn't fill a string column**<br><code>na.fill</code> only fills columns of a matching type. Use a dictionary per column.<br><br>
**⛔ Problem: <code>isNull</code> finds nothing, but the column looks empty**<br>Empty strings or placeholder text such as <code>N/A</code> aren't null. Standardize them with <code>na.replace</code> or a <code>when</code>.<br><br>
**⛔ Problem: SQL <code>NOT IN (subquery)</code> returns no rows**<br>The subquery contains a null. Use <code>NOT EXISTS</code> or a left anti join, or filter nulls out of the subquery.<br><br>
**⛔ Problem: <code>DIVIDE_BY_ZERO</code> error**<br>ANSI mode is on. Use <code>try_divide(a, b)</code> or <code>a / nullif(b, 0)</code>.<br><br>
<b>⛔ Problem: join returns fewer rows than expected</b><br>Null join keys never match with <code>=</code>. Investigate why keys are null, or use <code>eqNullSafe</code> if appropriate.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. How does Spark treat null in comparisons?</b><br>
It follows SQL three-valued logic. Any comparison with null returns null, which filters treat as not true. <code>&lt;=&gt;</code> (<code>eqNullSafe</code>) is the null-safe equality, which returns true for null versus null.<br><br>

<b>🎤 2. How do aggregate functions handle nulls?</b><br>
<code>sum</code>, <code>avg</code>, <code>min</code>, <code>max</code> and <code>count(col)</code> ignore nulls. <code>count(*)</code> counts all rows. So <code>avg</code> is the average of the known values, which may not be what the business wants.<br><br>

<b>🎤 3. What is the difference between <code>na.fill</code> and <code>coalesce</code>?</b><br>
<code>na.fill</code> replaces nulls with constants, per column or by type. <code>coalesce</code> is an expression that returns the first non-null of several columns or values, so it can fall back to other columns, not just constants.<br><br>

<b>🎤 4. What happens to null keys in a join?</b><br>
They never match with <code>=</code>, so they drop out of inner joins and get null partners in outer joins. Use <code>eqNullSafe</code> if nulls should match, but usually null keys are a data quality issue to fix.<br><br>

<b>🎤 5. Why does <code>NOT IN</code> with a null in the list return no rows?</b><br>
<code>x NOT IN (a, NULL)</code> means <code>x &lt;&gt; a AND x &lt;&gt; NULL</code>, and <code>x &lt;&gt; NULL</code> is null, so the whole condition is null or false and never true. <code>NOT EXISTS</code> or an anti join avoids this.<br><br>

<b>🎤 6. How do you count nulls in every column efficiently?</b><br>
One <code>select</code> with <code>count(when(col(c).isNull(), 1))</code> for each column. That is a single pass over the data, instead of one job per column.<br><br>

<b>🎤 7. How do you avoid division by zero under ANSI mode?</b><br>
<code>try_divide(a, b)</code> returns null, as does <code>a / nullif(b, 0)</code>.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. A table has 10 rows, and the <code>bonus</code> column is NULL in 4 of them and 100 in the others. What does <code>SELECT avg(bonus) FROM t</code> return?</b><br>
A. 60<br>
B. 100<br>
C. NULL<br>
D. 0<br>
<details><summary><b>Show answer</b></summary><b>B.</b> <code>avg</code> ignores NULLs, so it averages the 6 known values.</details><br><br>

<b>Q2. Which expression returns <code>phone</code> if it isn't NULL, otherwise <code>email</code>, otherwise <code>'no contact'</code>?</b><br>
A. <code>nvl2(phone, email, 'no contact')</code><br>
B. <code>coalesce(phone, email, 'no contact')</code><br>
C. <code>nullif(phone, email)</code><br>
D. <code>ifnull(phone, email, 'no contact')</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b> <code>coalesce</code> returns the first non-NULL argument. <code>ifnull</code> takes only two arguments.</details><br><br>

<b>Q3. Two tables are joined with <code>ON a.key = b.key</code>. Both have rows where <code>key</code> is NULL. What happens to those rows in an INNER JOIN?</b><br>
A. They match each other<br>
B. They are excluded<br>
C. The query fails<br>
D. They match every row<br>
<details><summary><b>Show answer</b></summary><b>B.</b> <code>NULL = NULL</code> is NULL, so the join condition is never true for them.</details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b><br>
Using <code>customers</code>:
<ol><li>Produce a null profile: for each column, the number <b>and percentage</b> of nulls, after turning <code>""</code>, <code>"N/A"</code> and <code>"null"</code> into real nulls</li><li>Drop rows without a <code>customer_id</code></li><li>Fill <code>orders</code> with 0, <code>name</code> with <code>"unknown"</code>, and <code>income</code> with the <b>average</b> income</li><li>Add <code>best_contact</code>: email, else mobile, else home phone, else <code>"none"</code></li><li>Calculate <code>income_per_order</code> without errors when orders is 0 or null</li><li>In SQL, show the result of <code>NULL &lt;&gt; 'x'</code> and explain it in a comment</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1. Standardize fake nulls in every string column, then profile
string_cols = [n for n, t in customers.dtypes if t == "string"]
std = customers.na.replace(["", "N/A", "null"], None, subset=string_cols)
total = std.count()
profile = std.select([F.count(F.when(F.col(c).isNull(), 1)).alias(c) for c in std.columns]).first().asDict()
spark.createDataFrame(
    [(c, n, round(n / total * 100, 1)) for c, n in profile.items()], "column STRING, nulls INT, pct DOUBLE"
).show()

# 2. Drop rows without a key
p = std.na.drop(subset=["customer_id"])

# 3. Fill with constants and a computed average
avg_income = p.agg(F.avg("income")).first()[0]
p = p.na.fill({"orders": 0, "name": "unknown", "income": avg_income})

# 4-5. Fallback contact and safe division
p = p.withColumns({
    "best_contact": F.coalesce("email", "mobile", "home_phone", F.lit("none")),
    "income_per_order": F.expr("try_divide(income, orders)"),
})
p.select("customer_id", "name", "orders", "income", "best_contact", "income_per_order").show(truncate=False)

# 6. NULL <> 'x' is NULL: comparing with an unknown value gives an unknown result
spark.sql("SELECT NULL <> 'x' AS result").show()

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>Null = unknown. Comparisons with null return null, and filters drop those rows</li><li>Aggregates ignore nulls, except <code>count(*)</code></li><li>Null join keys never match with <code>=</code>. <code>eqNullSafe</code> / <code>&lt;=&gt;</code> matches them</li><li><code>NOT IN</code> with a null in the list never returns true. Prefer <code>NOT EXISTS</code> / anti join</li><li>Standardize fake nulls (<code>""</code>, <code>N/A</code>) into real nulls early</li><li><code>na.fill</code> (type-matched), <code>coalesce</code> (first non-null), <code>na.drop</code> (<code>any</code>, <code>all</code>, <code>thresh</code>, <code>subset</code>)</li><li><code>try_divide</code> or <code>nullif(b, 0)</code> avoids divide-by-zero under ANSI mode</li><li>Profile nulls in one pass with <code>count(when(isNull))</code> per column</li></ul>
</div>

| Task | Code |
|---|---|
| Find | `F.col("c").isNull()`, `isNotNull()` |
| Count per column | `[F.count(F.when(F.col(c).isNull(), 1)).alias(c) for c in df.columns]` |
| Fill | `df.na.fill({"a": 0, "b": "x"})` |
| First non-null | `F.coalesce("a", "b", F.lit("x"))` |
| Drop | `df.na.drop(subset=["id"])` |
| Replace text | `df.na.replace(["", "N/A"], None, subset=["c"])` |
| Null-safe equal | `F.col("a").eqNullSafe(F.col("b"))`, SQL `a <=> b` |
| Safe divide | `try_divide(a, b)` |

## Git commit

```
git add 01_spark_basics/16_null_handling.ipynb
git commit -m "add 16 null handling notebook"
```